# 02 — VaR & Expected Shortfall: Three Methods Compared

**Value at Risk (VaR)** answers: *"What is the worst loss I should expect over a given horizon at a given confidence level?"*

**Expected Shortfall (ES)**, also called Conditional VaR, goes further: *"If losses exceed the VaR threshold, how bad do they get on average?"*

We compare three VaR estimation methods — **Historical**, **Parametric**, and **Monte Carlo** — because each makes different assumptions about return distributions, correlation structure, and tail behavior. Understanding where they agree and diverge is critical for robust risk management.

---

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from var_risk_engine.data import fetch_and_prepare
from var_risk_engine.var_historical import historical_var, historical_var_series
from var_risk_engine.var_parametric import parametric_var, parametric_var_from_cov
from var_risk_engine.var_montecarlo import montecarlo_var, montecarlo_var_with_details, simulate_gbm_paths
from var_risk_engine.expected_shortfall import expected_shortfall, es_parametric, compare_var_es
from var_risk_engine.covariance import sample_covariance, correlation_from_covariance

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# --- Fetch data and compute portfolio returns ---
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
prices = fetch_and_prepare(tickers)
returns = prices.pct_change().dropna()

weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
portfolio_returns = returns.values @ weights

print(f"Assets:       {tickers}")
print(f"Weights:      {weights}")
print(f"Observations: {len(portfolio_returns)}")
print(f"Mean return:  {portfolio_returns.mean():.5f}")
print(f"Std return:   {portfolio_returns.std():.5f}")

In [ ]:
# --- VaR / ES at 95% confidence: all three methods ---
alpha = 0.95

h_var = historical_var(portfolio_returns, alpha)
p_var = parametric_var(portfolio_returns, alpha)
mc_var, mc_details = montecarlo_var_with_details(portfolio_returns, alpha, n_simulations=100_000)

h_es  = expected_shortfall(portfolio_returns, alpha)
p_es  = es_parametric(portfolio_returns, alpha)

comparison = pd.DataFrame({
    "Method":       ["Historical", "Parametric", "Monte Carlo"],
    f"VaR ({alpha})":  [h_var, p_var, mc_var],
    f"ES ({alpha})":   [h_es,  p_es,  p_es],
})
comparison[f"ES - VaR"] = comparison[f"ES ({alpha})"] - comparison[f"VaR ({alpha})"]

print(f"\n{'='*60}")
print(f"  VaR & Expected Shortfall — 95% Confidence")
print(f"{'='*60}")
print(comparison.to_string(index=False, float_format="%.5f"))
print(f"{'='*60}")

In [ ]:
# --- Multi-confidence comparison ---
alphas = [0.90, 0.95, 0.975, 0.99]
var_es_table = compare_var_es(portfolio_returns, alphas)

print("\n  VaR & ES across confidence levels")
print("  " + "-" * 70)
print(var_es_table.to_string(float_format="%.5f"))

In [ ]:
# --- Portfolio return histogram with VaR lines and KDE ---
fig, ax = plt.subplots(figsize=(12, 6))

n, bins, patches = ax.hist(
    portfolio_returns, bins=80, density=True,
    alpha=0.5, color="#4c72b0", edgecolor="white", label="Realized returns"
)

# KDE overlay
xmin, xmax = ax.get_xlim()
x_kde = np.linspace(xmin, xmax, 500)
kde = stats.gaussian_kde(portfolio_returns)
ax.plot(x_kde, kde(x_kde), color="black", linewidth=1.8, label="KDE")

# VaR lines
colors = {"Historical": "#c44e52", "Parametric": "#dd8452", "Monte Carlo": "#55a868"}
var_vals = {"Historical": h_var, "Parametric": p_var, "Monte Carlo": mc_var}

for method, val in var_vals.items():
    ax.axvline(-val, color=colors[method], linestyle="--", linewidth=2,
               label=f"{method} VaR = {-val:.4f}")

ax.set_xlabel("Daily Portfolio Return")
ax.set_ylabel("Density")
ax.set_title("Portfolio Return Distribution with 95% VaR Thresholds")
ax.legend(loc="upper left", frameon=True, framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Monte Carlo simulation: GBM paths and return distribution ---
# Geometric Brownian Motion:  S_t = S_0 * exp((mu - sigma^2/2)*t + sigma*sqrt(t)*Z)

mu = portfolio_returns.mean()
sigma = portfolio_returns.std()
n_paths = 20_000

simulated_paths = simulate_gbm_paths(
    mu=mu,
    sigma=sigma,
    n_paths=n_paths,
    n_steps=1,
    S0=1.0
)

# Extract simulated 1-day returns
simulated_returns = simulated_paths[:, -1] / simulated_paths[:, 0] - 1.0

mc_sim_var = -np.percentile(simulated_returns, (1 - alpha) * 100)
mc_sim_es  = -simulated_returns[simulated_returns < -mc_sim_var].mean()

# --- Plot simulated return distribution ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: sample GBM paths
axes[0].plot(simulated_paths[:100].T, alpha=0.15, color="steelblue", linewidth=0.5)
axes[0].set_title(f"100 Sample GBM Paths (of {n_paths:,})\n$S_t = S_0 e^{{(\\mu - \\sigma^2/2)t + \\sigma\\sqrt{{t}}Z}}$")
axes[0].set_xlabel("Time Step")
axes[0].set_ylabel("Price")

# Right: simulated return distribution
ax = axes[1]
ax.hist(simulated_returns, bins=120, density=True, alpha=0.55,
        color="#4c72b0", edgecolor="white", label="Simulated returns")

x_kde = np.linspace(simulated_returns.min(), simulated_returns.max(), 500)
kde_sim = stats.gaussian_kde(simulated_returns)
ax.plot(x_kde, kde_sim(x_kde), color="black", linewidth=1.8, label="KDE")

# VaR and ES lines
ax.axvline(-mc_sim_var, color="#c44e52", linestyle="--", linewidth=2,
           label=f"MC VaR (95%) = {-mc_sim_var:.4f}")
ax.axvline(-mc_sim_es, color="#8172b2", linestyle="-.", linewidth=2,
           label=f"MC ES (95%)  = {-mc_sim_es:.4f}")

# Shaded tail region (losses beyond VaR)
tail_mask = x_kde < -mc_sim_var
ax.fill_between(x_kde, kde_sim(x_kde), where=tail_mask,
                color="#c44e52", alpha=0.25, label="Tail (beyond VaR)")

ax.set_xlabel("Simulated Daily Return")
ax.set_ylabel("Density")
ax.set_title(f"Monte Carlo Simulated Return Distribution\n({n_paths:,} paths, 1-day horizon)")
ax.legend(loc="upper left", frameon=True, framealpha=0.9)

plt.tight_layout()
plt.show()

print(f"\nSimulated VaR (95%): {-mc_sim_var:.5f}")
print(f"Simulated ES  (95%): {-mc_sim_es:.5f}")
print(f"ES - VaR gap:        {mc_sim_es - mc_sim_var:.5f}")

In [ ]:
# --- Rolling Historical VaR (250-day window) ---
window = 250

rolling_var = historical_var_series(portfolio_returns, alpha, window=window)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(rolling_var.index, rolling_var.values, color="#c44e52", linewidth=1.5,
        label=f"Rolling {window}-day Historical VaR (95%)")
ax.fill_between(rolling_var.index, 0, rolling_var.values,
                alpha=0.12, color="#c44e52")

ax.set_xlabel("Date")
ax.set_ylabel("VaR (positive = loss)")
ax.set_title(f"Rolling {window}-Day Historical VaR at 95% Confidence")
ax.legend(loc="upper left", frameon=True)

# Annotate max VaR
max_idx = rolling_var.idxmax()
max_val = rolling_var.max()
ax.annotate(f"Max: {max_val:.4f}\n({max_idx.strftime('%Y-%m-%d')})",
            xy=(max_idx, max_val), xytext=(30, 20),
            textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=10, color="#c44e52")

plt.tight_layout()
plt.show()

print(f"Mean rolling VaR:   {rolling_var.mean():.5f}")
print(f"Max  rolling VaR:   {max_val:.5f} on {max_idx.strftime('%Y-%m-%d')}")
print(f"Min  rolling VaR:   {rolling_var.min():.5f}")

In [ ]:
# --- Parametric VaR from covariance matrix: w^T Sigma w ---
cov_matrix = sample_covariance(returns)
corr_matrix = correlation_from_covariance(cov_matrix)

print("Covariance Matrix (annualized):")
print(cov_matrix.round(6).to_string())
print("\nCorrelation Matrix:")
print(corr_matrix.round(4).to_string())

# Portfolio variance: w^T @ Sigma @ w
w = np.array(weights)
Sigma = cov_matrix.values
port_var = w @ Sigma @ w
port_vol = np.sqrt(port_var)

z_95 = stats.norm.ppf(0.95)
param_var_manual = -(w @ returns.mean().values + z_95 * port_vol)

print(f"\nPortfolio variance (w^T Sigma w): {port_var:.8f}")
print(f"Portfolio volatility:              {port_vol:.5f}")
print(f"z-score at 95%:                    {z_95:.4f}")
print(f"Parametric VaR (manual):           {param_var_manual:.5f}")

# Verify with module function
param_var_cov = parametric_var_from_cov(returns, weights, alpha)
print(f"Parametric VaR (module):           {param_var_cov:.5f}")
print(f"Difference:                        {abs(param_var_manual - param_var_cov):.2e}")

# Cholesky verification
L = np.linalg.cholesky(Sigma)
print(f"\nCholesky factor L (lower triangular):")
print(L.round(6))

# Verify: L @ L^T should reconstruct Sigma
Sigma_reconstructed = L @ L.T
reconstruction_error = np.max(np.abs(Sigma - Sigma_reconstructed))
print(f"\nMax reconstruction error (L @ L^T vs Sigma): {reconstruction_error:.2e}")
print("Cholesky decomposition verified." if reconstruction_error < 1e-12 else "WARNING: Cholesky reconstruction error is large.")

---

## Mathematical Foundations

### 1. Historical VaR

$$
\text{VaR}_\alpha = -Q_{1-\alpha}(R)
$$

where $Q_{1-\alpha}(R)$ is the empirical $(1-\alpha)$-quantile of the return distribution. **No distributional assumption is made** — we simply sort realized returns and read off the appropriate percentile. This makes Historical VaR robust to fat tails and skewness, but entirely dependent on the available sample.

### 2. Parametric (Variance-Covariance) VaR

$$
\text{VaR}_\alpha = -\left(\mu + z_\alpha \cdot \sigma\right)
$$

where $\mu$ and $\sigma$ are the mean and standard deviation of portfolio returns, and $z_\alpha = \Phi^{-1}(\alpha)$ is the inverse standard normal CDF. This **assumes returns are normally distributed**, which tends to understate tail risk. The portfolio variance is computed as:

$$
\sigma_p^2 = \mathbf{w}^T \Sigma \mathbf{w}
$$

For simulation, the **Cholesky decomposition** $\Sigma = L L^T$ generates correlated random draws $\mathbf{x} = L \mathbf{z}$ where $\mathbf{z} \sim \mathcal{N}(0, I)$.

### 3. Monte Carlo VaR

Returns are simulated via **Geometric Brownian Motion (GBM)**:

$$
S_t = S_0 \exp\!\left[\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma\sqrt{t}\,Z\right], \quad Z \sim \mathcal{N}(0,1)
$$

VaR is then the empirical quantile of the simulated loss distribution. Monte Carlo is flexible and can accommodate any distribution or path dependency, but is computationally intensive and sensitive to parameter estimates.

### 4. Expected Shortfall (ES)

$$
\text{ES}_\alpha = \mathbb{E}\!\left[L \;\middle|\; L > \text{VaR}_\alpha\right] = -\frac{1}{1-\alpha}\int_0^{1-\alpha} Q_p(R)\,dp
$$

ES is the **conditional expectation of losses beyond VaR**. Unlike VaR, ES is a **coherent risk measure** — it satisfies:

| Property | Definition |
|---|---|
| **Sub-additivity** | $\text{ES}(X + Y) \leq \text{ES}(X) + \text{ES}(Y)$ |
| **Monotonicity** | $X \leq Y \implies \text{ES}(X) \geq \text{ES}(Y)$ |
| **Translation invariance** | $\text{ES}(X + c) = \text{ES}(X) - c$ |
| **Positive homogeneity** | $\text{ES}(\lambda X) = \lambda\,\text{ES}(X)$ for $\lambda > 0$ |

VaR fails sub-additivity, meaning a portfolio's VaR can exceed the sum of its parts — a fundamental flaw that ES corrects.

---

## Key Findings

### Which method gives the highest / lowest VaR?

- **Parametric VaR** is typically the **lowest** of the three because the normal distribution underestimates tail probability. Real equity returns exhibit fat tails (excess kurtosis) and negative skewness that a Gaussian model ignores.
- **Historical VaR** captures the actual return distribution, including crashes and rallies present in the sample. It can be higher or lower than Parametric depending on the lookback period.
- **Monte Carlo VaR** falls between Historical and Parametric when using GBM with normal innovations, since it essentially resamples from a fitted normal distribution.

### When do the methods diverge most?

- **Fat-tailed or skewed distributions**: The gap between Historical and Parametric VaR widens during or after market stress (e.g., 2008, 2020) when realized returns deviate sharply from normality.
- **Small sample sizes**: Historical VaR becomes noisy with short lookback windows; Parametric and Monte Carlo are more stable but rely on their distributional assumptions.
- **High confidence levels (99%+)**: Differences amplify in the extreme tail where data is sparse and model assumptions dominate.

### Why is ES always greater than or equal to VaR?

By definition, ES averages all losses **beyond** the VaR threshold. Since every loss in that tail is at least as large as VaR itself, the average must satisfy:

$$
\text{ES}_\alpha \geq \text{VaR}_\alpha \quad \forall\, \alpha \in (0, 1)
$$

Equality holds only in degenerate cases (e.g., a point mass at VaR). For any continuous or dispersed loss distribution, ES is strictly greater. This makes ES a more conservative and informative measure of tail risk — which is precisely why Basel III replaced VaR with ES for market risk capital requirements.